**Company House Table **creation****

In [0]:
from pyspark.sql import functions as F

raw_path = "/Volumes/company-risk-intelligence-platform/bronze/raw_data/companies_house/filing_history/"

df = (
    spark.read
    .option("multiline", "true")   # if JSON spans multiple lines
    .json(raw_path)
)

display(df)

In [0]:
df.write.mode("overwrite").saveAsTable("company_risk_intelligence_platform.bronze.companu_house_filing_history")

In [0]:
display(spark.sql("select * from company_risk_intelligence_platform.bronze.companu_house_filing_history"))

In [0]:
from pyspark.sql.functions import current_timestamp, col

df = df.withColumn("last_update_ts", current_timestamp()) \
             .withColumn("file_path", col("_metadata.file_path"))
df.display()

In [0]:
base_path = "/Volumes/company_risk_intelligence_platform/bronze/raw_data/companies_house/"

table_mapping = {
    "ch_filing_history": "filing_history",
    "ch_overview": "overview",
    "ch_people": "people"
}

catalog = "company_risk_intelligence_platform"
schema = "bronze"

def add_metadata_columns(df):
    return df 
        .withColumn("last_update_ts", current_timestamp()) \
        .withColumn("file_path", col("_metadata.file_path"))

for table, folder in table_mapping.items():
    print(f"Processing {table}")

    input_path = base_path + folder + "/"

    df = spark.read.option("multiline", "true").json(input_path)

    df = add_metadata_columns(df)

    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{catalog}.{schema}.{table}")

print("All tables processed")

In [0]:
tables_df = spark.sql("""
SHOW TABLES IN company_risk_intelligence_platform.bronze
""")

display(tables_df)

In [0]:
display(spark.sql("select * from company_risk_intelligence_platform.bronze.ch_overview"))

In [0]:
display(spark.sql("select * from company_risk_intelligence_platform.bronze.ch_people"))

In [0]:
display(spark.sql("select * from company_risk_intelligence_platform.bronze.ch_filing_history"))

**YF Table Creation**

In [0]:
from pyspark.sql.functions import current_timestamp, col

base_path = "/Volumes/company_risk_intelligence_platform/bronze/raw_data/yfinance/"

table_mapping = {
    "yf_stock": "stock",
    "yf_news": "news",
    "yf_income_statement": "income_statement",
    "yf_balance_sheet": "balance_sheet",
    "yf_cashflow": "cashflow",
    "yf_info": "info"
}

catalog = "company_risk_intelligence_platform"
schema = "bronze"

def add_metadata_columns(df):
    return (
        df.withColumn("last_update_ts", current_timestamp())
          .withColumn("file_path", col("_metadata.file_path"))
    )

for table, folder in table_mapping.items():

    input_path = f"{base_path}{folder}/"

    try:

        print(f"Processing {table}")

        df = (
            spark.read
                .option("multiline", "true")
                .option("recursiveFileLookup", "true")
                .json(input_path)
        )

        df = add_metadata_columns(df)

        (
            df.write.format("delta")
              .mode("overwrite")
              .saveAsTable(f"{catalog}.{schema}.{table}")
        )

        print(f"Loaded {table}")

    except Exception as e:
        print(f"Skipping {table}: {e}")

print("All Yahoo Finance bronze tables processed")

In [0]:
import pandas as pd
df= pd.read_json("/Volumes/company_risk_intelligence_platform/bronze/raw_data/yfinance/stock/AML_L.json")
df.display()